# Day 23: Image‑to‑Image & Inpainting with Diffusers

In [ ]:
import torch
from diffusers import StableDiffusionImg2ImgPipeline, StableDiffusionInpaintPipeline
from PIL import Image
import requests
from io import BytesIO
import numpy as np
import matplotlib.pyplot as plt

## 1. Image‑to‑Image (transform an existing image)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
img2img_pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5", torch_dtype=torch.float16
).to(device)
img2img_pipe.safety_checker = None

In [ ]:
# Load a starting image (landscape)
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/beach.png"
init_img = Image.open(requests.get(url, stream=True).raw).convert("RGB")
init_img = init_img.resize((512, 512))
display(init_img)

prompt = "a fantasy landscape with a castle on a hill, highly detailed"
strength = 0.75  # controls how much the image changes (0 = no change, 1 = full change)

with torch.no_grad():
    result = img2img_pipe(
        prompt=prompt,
        image=init_img,
        strength=strength,
        guidance_scale=7.5
    ).images[0]

result

## 2. Inpainting (edit specific regions)
Use a separate pipeline that supports a mask.

In [ ]:
inpaint_pipe = StableDiffusionInpaintPipeline.from_pretrained(
    "runwayml/stable-diffusion-inpainting", torch_dtype=torch.float16
).to(device)
inpaint_pipe.safety_checker = None

In [ ]:
# Use a photo of a cat, create a mask for its eyes
cat_url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/cats.png"
cat_img = Image.open(requests.get(cat_url, stream=True).raw).convert("RGB").resize((512, 512))

# Create a white mask (0 = keep, 255 = regenerate)
mask = Image.new("L", (512, 512), 0)
# Draw a circle over the eyes area (approximate)
from PIL import ImageDraw
draw = ImageDraw.Draw(mask)
draw.ellipse((200, 180, 310, 280), fill=255)  # eye region

display(cat_img)
display(mask)

prompt = "bright blue glowing anime eyes"

with torch.no_grad():
    inpainted = inpaint_pipe(
        prompt=prompt,
        image=cat_img,
        mask_image=mask,
        num_inference_steps=30,
        guidance_scale=7.0
    ).images[0]

inpainted

## 3. Experiment with denoising strength in img2img
Lower strength = closer to original; higher = more creative.

In [ ]:
for s in [0.2, 0.5, 0.8]:
    out = img2img_pipe(prompt="a futuristic city", image=init_img, strength=s).images[0]
    print(f"Strength {s}")
    display(out)